# w9_save_4096_tag.ipynb — save the tag at big anchors (4096-only trio)

Diagnosis (data): the tag decline at large caps tracks the ANCHOR SUPPLY, not
I -- at 512 i2ce's tag BEATS ce's (.728 vs .702); the C-term doesn't fix it
(i2cce@2048 .706 = i2ce .707); the mq queue (EMA-shadow keys, no grad) at the
SAME cap/I/C restores it (.736). Culprit: CE's gradient flows through Zg and
games the pack embeddings off the content manifold late in training (i2ce@4096
ZS tag decays within training: .726 -> .709).

Three fixes, tested at 4096 / 2000ep / ZS-only:
- **i2sgce** (5a): stop-grad gallery via torch.no_grad() -- anchors can't be
  gamed; measured VRAM 4096: 45G -> ~18G, 8192: 89G -> ~36G, so the no-grad
  arm ALSO runs at **8192** (full-coverage anchors on 80G, a first).
- **i2esce** (5b): EMA-shadow gallery (m=.99, lag 100 steps ~ 6 ep) -- stop-
  grad + target smoothing; same fixed points as sg, damped transients.
- **i2q2ce** (1): soft-band I, lambda*(1-cos)^2 -- near-aligned dead band
  (5x softer at cos .9) preserves per-view individuality; tests the I-side
  hypothesis on its own merits.
i2ce@4096 rides as the disease reference (scale owns it; claims coordinate).
Run on an **A100 80G** pod; ~1 day/cell at 4096 pace, packs by measured cost.


In [ ]:
# constants
import os

REPO = os.path.abspath("..")   # this release folder (contains Pod/ and VICReg_review/)
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # shared campaign out dir

# VRAM scheduler knobs: budget = SAFETY x free - RESERVE. Costs are measured
# PER (arm, cap): grad and no-grad arms at the same cap differ ~2.5x.
SAFETY = 0.85
RESERVE_GIB = 1.5

# (arm, cap, epochs)
FLASH = [
    ("wcle_i2sgce_icetf", 4096, 2000),   # 5a stop-grad gallery
    ("wcle_i2esce_icetf", 4096, 2000),   # 5b EMA-shadow gallery
    ("wcle_i2q2ce_icetf", 4096, 2000),   # 1  soft-band I (lambda*(1-cos)^2)
    ("wcle_i2sgce_icetf", 8192, 2000),   # 5a's VRAM gift: full 8192 anchors
    ("wcle_i2ce_icetf", 4096, 2000),     # disease reference (skips if done)
]
os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(FLASH)} cells")


In [ ]:
# Local setup (release build: the code ships with this folder -- no
# repository synchronisation is needed or performed).
import importlib.util
import os
import sys
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip -q install scikit-learn scipy
        break
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")

In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# VRAM scheduler, per-(arm,cap) measured costs (sg/es are ~2.5x lighter
# than grad-gallery arms at the same cap -- one cost per cap would misplan).
import os, subprocess, tempfile, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
# measure runs write to a LOCAL scratch dir: with the real job name on the
# SHARED volume, a measure could load another machine's resume bundle
# (start_ep >= 1 -> zero steps -> no cost file) or race its zs_traj writes.
MEAS_OUT = os.path.join(tempfile.gettempdir(), "w9_measure_out")
os.makedirs(MEAS_OUT, exist_ok=True)
gpus = J.detect_gpus()

def _smi_mib(field, g):
    out = subprocess.check_output(
        ["nvidia-smi", f"--query-gpu={field}", "--format=csv,noheader,nounits",
         "-i", str(g)]).decode().strip().split("\n")[0]
    return int(out) * 2**20

free = {g: _smi_mib("memory.free", g) for g in gpus}
budget = {g: int(free[g] * SAFETY - RESERVE_GIB * 2**30) for g in gpus}
print(f"[vram] budgets {[f'{budget[g] / 2**30:.0f}G' for g in gpus]}")

todo0 = []
for arm, cap, ep in FLASH:
    nm = J.fs_label(arm, cap, False, 0, "clean", 16)
    if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{ep}.npz").exists():
        print(f"[skip] {nm} at {ep}"); continue
    todo0.append((arm, cap, ep, nm))

# warmup: one measured cost per DISTINCT (arm, cap) among the not-done cells
cost = {}
for arm, cap in sorted({(a, c) for a, c, _e, _n in todo0}):
    tf = Path(tempfile.gettempdir()) / f"w9vram_{arm}_{cap}.txt"
    tf.unlink(missing_ok=True)
    cmd = ["python", "-u", J.FS_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           MEAS_OUT, "--repo", REPO, "--arm", arm, "--anchor-cap", str(cap),
           "--epochs", "1", "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--measure-vram", str(tf)]
    print(f"[warmup] {arm}@{cap} ...", flush=True)
    with open(logd / f"measure_{arm}_g{cap}.log", "w") as fh:
        subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0]))
    cost[(arm, cap)] = int(tf.read_text()) if tf.exists() else budget[gpus[0]] + 1
    print(f"[warmup] {arm}@{cap}: {cost[(arm, cap)] / 2**30:.2f}G", flush=True)

todo = sorted(((a, c, e, n, cost[(a, c)]) for a, c, e, n in todo0),
              key=lambda x: -x[4])
now_used = {g: 0 for g in gpus}
fails = []
cv = threading.Condition()

def run_job(g, arm, cap, ep, nm, c):
    try:
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
        cmd = ["python", "-u", J.FS_WORKER, "--data-dir", DATA_DIR, "--out-dir",
               OUT_DIR, "--repo", REPO, "--arm", arm, "--anchor-cap", str(cap),
               "--epochs", str(ep), "--ckpt-every", str(J.CKPT_EVERY),
               "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
               "--topup-seeds", str(J.TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        t0 = time.time()
        with open(logd / f"{arm}_g{cap}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
        if p.returncode != 0:
            (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
        print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} @{ep} "
              f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)
    finally:
        with cv:
            now_used[g] -= c
            cv.notify_all()

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
active = []
with cv:
    pending = list(todo)
    while pending or any(t.is_alive() for t in active):
        prog = False
        i = 0
        while i < len(pending):
            arm, cap, ep, nm, c = pending[i]
            fit = [g for g in gpus if now_used[g] + c <= budget[g] or now_used[g] == 0]
            if not fit:
                i += 1; continue
            g = min(fit, key=lambda g: now_used[g])
            now_used[g] += c
            th = threading.Thread(target=run_job, args=(g, arm, cap, ep, nm, c),
                                  daemon=True)
            active.append(th); th.start(); pending.pop(i)
            print(f"[sched] {nm} -> gpu{g} ({c / 2**30:.1f}G, used "
                  f"{now_used[g] / 2**30:.1f}/{budget[g] / 2**30:.0f}G)", flush=True)
            prog = True
        active = [t for t in active if t.is_alive()]
        if not prog:
            cv.wait(timeout=3)
stop_evt.set()
for t in active:
    t.join()
print(f"drained; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)


In [ ]:
# Readout: did the tag survive 4096? ZSbest-primary; tag(neu/non) is THE
# column. References: the disease row (i2ce@4096), the healthy small cap
# (i2ce@512), the C-attempt (i2cce@2048) and the queue cure (mq@2048).
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
ROWS = [("wcle_i2sgce_icetf_g4096", "i2sgce@4096 (stop-grad)"),
        ("wcle_i2esce_icetf_g4096", "i2esce@4096 (EMA shadow)"),
        ("wcle_i2q2ce_icetf_g4096", "i2q2ce@4096 (soft-band I)"),
        ("wcle_i2sgce_icetf_g8192", "i2sgce@8192 (the gift)"),
        ("wcle_i2ce_icetf_g4096", "i2ce@4096 (disease ref)"),
        ("wcle_i2ce_icetf", "i2ce@512 (healthy ref)"),
        ("wcle_i2cce_icetf_g2048", "i2cce@2048 (+C, failed fix)"),
        ("wcle_mq3072i2cce_icetf_g2048", "mq@2048 (queue cure)")]
def _row(lab, nm):
    zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    zp = Path(OUT_DIR) / f"zs_traj_{nm}_fp.json"
    ft = Path(OUT_DIR) / f"ft4var_{nm}_fp_best.json"
    if zb.exists():
        d = json.loads(zb.read_text())
        m4z = np.mean([d["nm_" + v] for v in VORD])
        line = (f"{lab:30s} ZSbest@ep{d['best_ep']:>4}(val) "
                + " ".join(f"{v[:3]}:{d['nm_' + v]:.3f}" for v in VORD)
                + f" m4z:{m4z:.3f} tag:{d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    elif zp.exists():
        tr = json.loads(zp.read_text())
        eps = sorted(tr, key=lambda k: int(k[2:]))
        pk = max(eps, key=lambda k: tr[k]["nm_neutral"])
        line = (f"{lab:30s} ZS test-peak*@{pk[2:]:>4} neu {tr[pk]['nm_neutral']:.3f}"
                f" non {tr[pk]['nm_noname']:.3f}"
                f" tag {tr[pk]['tag_neutral']:.3f}/{tr[pk]['tag_noname']:.3f}")
    else:
        return f"{lab:30s} (pending)"
    if ft.exists():
        d2 = json.loads(ft.read_text())
        m4 = np.mean([np.mean([x[v]["h1"] for x in d2["per_seed"]]) for v in VORD])
        line += f" | FT m4 {m4:.3f}"
    return line

for arm, lab in ROWS:
    print(_row(lab, f"w9_{arm}"))


In [ ]:
# AUTO-STOP removed in the release build: stopping the machine is cloud-
# provider tooling, not part of the experiment. All results are already on
# the shared volume when the run cells finish.
print("run complete -- results are in", OUT_DIR)